In [2]:
"""
Run this from the volatilityregimes/ root directory:

    python extract_stats.py

It prints every number needed to update project1_volatility.tex
with grounded, data-derived values.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew, kurtosis, jarque_bera, shapiro
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

ROOT = Path().resolve().parent
mxn  = pd.read_csv(ROOT / 'data/raw/mxn_usd.csv',  index_col='Date', parse_dates=True)
macro = pd.read_csv(ROOT / 'data/raw/macro.csv', index_col=0, parse_dates=True)

returns = np.log(mxn['MXN_USD'] / mxn['MXN_USD'].shift(1)).dropna()

# --- basic stats ---
mu         = returns.mean()
sigma      = returns.std()
emp_skew   = skew(returns)
emp_kurt   = kurtosis(returns, fisher=True)   # excess kurtosis

# --- tests ---
jb_stat, jb_p                         = jarque_bera(returns)
lm_stat, lm_p, _, _                   = het_arch(returns, nlags=10)
lb_r2   = acorr_ljungbox(returns**2,  lags=[10], return_df=True)
lb_r    = acorr_ljungbox(returns,     lags=[10], return_df=True)
adf_stat, adf_p, _, _, adf_cv, _      = adfuller(returns)

# --- rolling vol ---
vol_30 = returns.rolling(30).std()  * np.sqrt(252)
vol_90 = returns.rolling(90).std()  * np.sqrt(252)

# --- date range ---
start = returns.index[0].date()
end   = returns.index[-1].date()
n_obs = len(returns)

# --- crisis peaks (approx 30-day vol around known dates) ---
crises = {
    'Tequila 1994' : ('1994-11-01', '1995-03-31'),
    'Dot-com 2001' : ('2001-08-01', '2001-12-31'),
    'GFC 2008'     : ('2008-08-01', '2009-03-31'),
    'Trump 2016'   : ('2016-10-01', '2017-01-31'),
    'COVID 2020'   : ('2020-02-01', '2020-06-30'),
}
crisis_peaks = {}
for name, (s, e) in crises.items():
    window = vol_30.loc[s:e]
    if len(window):
        crisis_peaks[name] = window.max()

print("=" * 60)
print("NUMBERS FOR project1_volatility.tex")
print("=" * 60)
print(f"\n--- SAMPLE ---")
print(f"Start date       : {start}")
print(f"End date         : {end}")
print(f"Observations (n) : {n_obs:,}")

print(f"\n--- RETURN DISTRIBUTION ---")
print(f"Mean daily return: {mu:.6f}")
print(f"Daily std dev    : {sigma:.6f}")
print(f"Annualised vol   : {sigma * np.sqrt(252):.4f}  ({sigma*np.sqrt(252):.1%})")
print(f"Skewness         : {emp_skew:.4f}")
print(f"Excess kurtosis  : {emp_kurt:.4f}")
print(f"  → implied Student-t df ≈ {6/emp_kurt + 4:.1f}  (rough: nu ≈ 6/K + 4)")

print(f"\n--- JARQUE-BERA TEST ---")
print(f"JB statistic : {jb_stat:,.1f}")
print(f"JB p-value   : {jb_p:.2e}")

print(f"\n--- ARCH-LM TEST (10 lags) ---")
print(f"LM statistic : {lm_stat:.4f}")
print(f"LM p-value   : {lm_p:.2e}")

print(f"\n--- LJUNG-BOX TEST ---")
print(f"Q(10) on r_t   : stat={lb_r['lb_stat'].iloc[0]:.4f}   p={lb_r['lb_pvalue'].iloc[0]:.4f}")
print(f"Q(10) on r_t^2 : stat={lb_r2['lb_stat'].iloc[0]:.4f}  p={lb_r2['lb_pvalue'].iloc[0]:.2e}")

print(f"\n--- ADF TEST ---")
print(f"ADF statistic : {adf_stat:.4f}")
print(f"ADF p-value   : {adf_p:.4f}")
print(f"Critical 1%   : {adf_cv['1%']:.4f}")

print(f"\n--- ROLLING VOLATILITY ---")
print(f"Median annualised vol (90-day) : {vol_90.median():.1%}")
print(f"Max 30-day vol (full sample)   : {vol_30.max():.1%}")
for name, peak in crisis_peaks.items():
    print(f"  Peak during {name:<20}: {peak:.1%}")

print(f"\n--- AUTOCORRELATION ---")
print(f"ACF r_t   lag 1 : {returns.autocorr(1):.4f}")
print(f"ACF r_t^2 lag 1 : {(returns**2).autocorr(1):.4f}")
print(f"ACF r_t^2 lag 5 : {(returns**2).autocorr(5):.4f}")
print(f"ACF r_t^2 lag 10: {(returns**2).autocorr(10):.4f}")

print("\n" + "=" * 60)
print("Paste this output back to Claude.")
print("=" * 60)

NUMBERS FOR project1_volatility.tex

--- SAMPLE ---
Start date       : 2000-01-04
End date         : 2026-03-06
Observations (n) : 6,588

--- RETURN DISTRIBUTION ---
Mean daily return: 0.000097
Daily std dev    : 0.006842
Annualised vol   : 0.1086  (10.9%)
Skewness         : 0.7792
Excess kurtosis  : 10.3022
  → implied Student-t df ≈ 4.6  (rough: nu ≈ 6/K + 4)

--- JARQUE-BERA TEST ---
JB statistic : 29,801.1
JB p-value   : 0.00e+00

--- ARCH-LM TEST (10 lags) ---
LM statistic : 1463.3399
LM p-value   : 2.09e-308

--- LJUNG-BOX TEST ---
Q(10) on r_t   : stat=33.6128   p=0.0002
Q(10) on r_t^2 : stat=3663.9767  p=0.00e+00

--- ADF TEST ---
ADF statistic : -26.6768
ADF p-value   : 0.0000
Critical 1%   : -3.4313

--- ROLLING VOLATILITY ---
Median annualised vol (90-day) : 9.1%
Max 30-day vol (full sample)   : 45.4%
  Peak during Dot-com 2001        : 9.8%
  Peak during GFC 2008            : 45.4%
  Peak during Trump 2016          : 28.9%
  Peak during COVID 2020          : 41.4%

--- AUTO